In [ ]:
import pandas as pd
import numpy as np
import re
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge

1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.

In [ ]:
data_train = pd.read_csv('salary-train.csv')
data_test  = pd.read_csv('salary-test-mini.csv')
print(data_train.shape, data_test.shape)
data_train.head(2)

2. Проведите предобработку:
- Приведите тексты к нижнему регистру.
- Замените все, кроме букв и цифр, на пробелы (`re.sub('[^a-zA-Z0-9]', ' ', text.lower())`).
- Примените TfidfVectorizer (min_df=5) для преобразования текстов в векторы признаков.
- Замените пропуски в столбцах LocationNormalized и ContractTime на строку 'nan'.
- Примените DictVectorizer для one-hot-кодирования категориальных признаков.
- Объедините все признаки в одну матрицу с помощью scipy.sparse.hstack.

In [ ]:
data_train['FullDescription'] = data_train['FullDescription'].apply(
    lambda t: re.sub('[^a-zA-Z0-9]', ' ', t.lower())
)
data_test['FullDescription'] = data_test['FullDescription'].apply(
    lambda t: re.sub('[^a-zA-Z0-9]', ' ', t.lower())
)

tfidf = TfidfVectorizer(min_df=5)
X_train_text = tfidf.fit_transform(data_train['FullDescription'])
X_test_text  = tfidf.transform(data_test['FullDescription'])
print('TF-IDF shape:', X_train_text.shape)

In [ ]:
data_train['LocationNormalized'] = data_train['LocationNormalized'].fillna('nan')
data_train['ContractTime']       = data_train['ContractTime'].fillna('nan')
data_test['LocationNormalized']  = data_test['LocationNormalized'].fillna('nan')
data_test['ContractTime']        = data_test['ContractTime'].fillna('nan')

enc = DictVectorizer()
X_train_categ = enc.fit_transform(
    data_train[['LocationNormalized', 'ContractTime']].to_dict('records')
)
X_test_categ = enc.transform(
    data_test[['LocationNormalized', 'ContractTime']].to_dict('records')
)

X_train = scipy.sparse.hstack([X_train_text, X_train_categ])
X_test  = scipy.sparse.hstack([X_test_text,  X_test_categ])
print('Итоговая матрица признаков:', X_train.shape)

3. Обучите гребневую регрессию с параметром alpha=1. Целевая переменная записана в столбце SalaryNormalized.

In [ ]:
y_train = data_train['SalaryNormalized']

ridge = Ridge(alpha=1)
ridge.fit(X_train, y_train)

4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv. Значения полученных прогнозов являются ответом на задание.

In [ ]:
predictions = ridge.predict(X_test)
print('Прогнозы:', [round(float(p), 2) for p in predictions])
print('Ответ:', ' '.join(str(round(float(p), 2)) for p in predictions))